# Evaluate external diagnosis predictions

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_diagnosis_label
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Read in true labels


In [ ]:
readfiletrue = paths.data("Update/ExternalValidation/True/external_sample100_notes_true_S1.xlsx")
true_result = pd.read_excel(readfiletrue)

In [ ]:
true_result.head()

In [ ]:
len(true_result)

## Read in already generated results


In [ ]:
readfileresults = paths.data("Update/ExternalValidation/Results/phase1_fewshot_binary.xlsx")
raw_predictions = pd.read_excel(readfileresults)

In [ ]:
llm_result = raw_predictions[["SMMDx_LLM_Binary"]]

In [ ]:
llm_result.head()

## Combine true and result data for performance


In [ ]:
final = pd.concat([true_result.reset_index(drop=True), llm_result.reset_index(drop=True)], axis=1)

## POST PROCESSING


In [ ]:
llm_columns = ["SMMDx_LLM_Binary", "smm_label_true"]
final = add_parsed_labels(final, llm_columns, parse_diagnosis_label)

In [ ]:
final.head(5)

## Note-level BINARY CLASSIFICATION PERFORMANCE


In [ ]:
conditions = [
    (final["post_smm_label_true"] == "Yes") & (final["post_SMMDx_LLM_Binary"] == "Yes"),
    (final["post_smm_label_true"] == "No") & (final["post_SMMDx_LLM_Binary"] == "No"),
    (final["post_smm_label_true"] == "Yes") & (final["post_SMMDx_LLM_Binary"] == "No"),
    (final["post_smm_label_true"] == "No") & (final["post_SMMDx_LLM_Binary"] == "Yes"),
]
choices = ["TP", "TN", "FN", "FP"]
final["Outcome1_Binary"] = np.select(conditions, choices, default="Unknown")

In [ ]:
final["y_true"] = final["post_smm_label_true"].str.lower().map({"yes": 1, "no": 0})
final["y_pred"] = final["post_SMMDx_LLM_Binary"].str.lower().map({"yes": 1, "no": 0})

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["Non-SMM", "SMM"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)

## Patient-level BINARY CLASSIFICATION PERFORMANCE


In [ ]:
final["post_smm_label_true2"] = np.where(final["post_smm_label_true"] == "Yes", 1, 0)

In [ ]:
ptlevel_true = final.groupby("pat_mrn_id")[["post_smm_label_true2"]].sum().reset_index()

In [ ]:
ptlevel_true.head()

In [ ]:
ptlevel_true["post_smm_label_true_Pt"] = np.where(
    ptlevel_true["post_smm_label_true2"] >= 1, "Yes", "No"
)
ptlevel_true2 = ptlevel_true.drop(columns="post_smm_label_true2")

In [ ]:
len(ptlevel_true2)

In [ ]:
final_pos = final[final["post_SMMDx_LLM_Binary"] == "Yes"]

In [ ]:
df_sorted = final_pos.sort_values(["pat_mrn_id"])
final_pos2 = df_sorted.drop_duplicates(subset="pat_mrn_id")

In [ ]:
len(final_pos2)

In [ ]:
final_pos2.loc[:, "SMMDx_LLM_Binary_Pt"] = "Yes"

In [ ]:
final_pos = final_pos2[["pat_mrn_id", "SMMDx_LLM_Binary_Pt"]]

In [ ]:
ptlevel_result = ptlevel_true2.merge(final_pos, on="pat_mrn_id", how="left")
ptlevel_result["SMMDx_LLM_Binary_Pt"] = ptlevel_result["SMMDx_LLM_Binary_Pt"].fillna("No")

In [ ]:
conditions = [
    (ptlevel_result["post_smm_label_true_Pt"] == "Yes")
    & (ptlevel_result["SMMDx_LLM_Binary_Pt"] == "Yes"),
    (ptlevel_result["post_smm_label_true_Pt"] == "No")
    & (ptlevel_result["SMMDx_LLM_Binary_Pt"] == "No"),
    (ptlevel_result["post_smm_label_true_Pt"] == "Yes")
    & (ptlevel_result["SMMDx_LLM_Binary_Pt"] == "No"),
    (ptlevel_result["post_smm_label_true_Pt"] == "No")
    & (ptlevel_result["SMMDx_LLM_Binary_Pt"] == "Yes"),
]
choices = ["TP", "TN", "FN", "FP"]
ptlevel_result["Outcome1_Binary_Pt"] = np.select(conditions, choices, default="Unknown")

In [ ]:
ptlevel_result["y_true"] = (
    ptlevel_result["post_smm_label_true_Pt"].str.lower().map({"yes": 1, "no": 0})
)
ptlevel_result["y_pred"] = (
    ptlevel_result["SMMDx_LLM_Binary_Pt"].str.lower().map({"yes": 1, "no": 0})
)

In [ ]:
y_true = ptlevel_result["y_true"]
y_pred = ptlevel_result["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["Non-SMM", "SMM"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)